# Aircraft Defect Detection - Colab GPU Training Notebook

이 노트북은 프로젝트의 로컬 학습 흐름을 Google Colab GPU 환경으로 옮긴 버전입니다.

**실행 결과**
- YOLOv8n 기반 aircraft defect detector 재학습
- `best.pt` 모델 가중치 생성
- 학습 경과 발표용 차트 PNG 생성
- confusion matrix, PR/F1 curve, sample prediction 이미지 정리
- 발표 자료에 붙여 넣기 좋은 산출물 ZIP 생성

**중요**
- Colab 메뉴에서 `Runtime > Change runtime type > T4 GPU` 또는 다른 GPU를 선택한 뒤 실행하세요.
- GPU에서 학습한 `best.pt`는 CPU에서도 추론할 수 있습니다. 다만 CPU 추론은 GPU보다 느립니다.
- 이 노트북에는 Roboflow 데이터 다운로드 API 키가 포함되어 있습니다. 외부 공유 전에는 키를 제거하세요.

## 1. Roboflow API Key Constant

데이터셋 다운로드에 쓰는 Roboflow API 키를 별도 상수로 분리했습니다. 외부 공유 전에는 이 셀의 값을 제거하세요.


In [ ]:
ROBOFLOW_API_KEY_VALUE = '1MeMAa0KjnstYnI0uCan'


## 2. Runtime and Training Configuration

프로젝트 README의 로컬 학습 설정을 Colab 기준 경로로 변환했습니다. 필요하면 `EPOCHS`, `BATCH`, `IMGSZ`, `PATIENCE`만 바꿔서 재학습하면 됩니다.

In [ ]:
from pathlib import Path
import os
import sys
import time
import json
import shutil
import zipfile
import random

ROOT = Path("/content/aircraft_defect_training")
DATASET_DIR = ROOT / "dataset"
RUNS_DIR = ROOT / "training" / "runs"
RUN_NAME = "aircraft-v1"
ASSET_DIR = Path("/content/presentation_assets")

ROBOFLOW_API_KEY = ROBOFLOW_API_KEY_VALUE
ROBOFLOW_WORKSPACE = "university-of-technology-sydney-21uto"
ROBOFLOW_PROJECT = "aircraft-defect-detection"
ROBOFLOW_VERSION = "latest"  # use an integer such as 3 to pin a version

BASE_MODEL = "yolov8n.pt"
EPOCHS = 20
BATCH = 16
IMGSZ = 640
PATIENCE = 5
SEED = 42
CONF_THRESHOLD = 0.10
IOU_THRESHOLD = 0.70

for path in [ROOT, DATASET_DIR, RUNS_DIR, ASSET_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"RUNS_DIR={RUNS_DIR}")
print(f"ASSET_DIR={ASSET_DIR}")
print(f"Training config: model={BASE_MODEL}, epochs={EPOCHS}, batch={BATCH}, imgsz={IMGSZ}, patience={PATIENCE}")

## 3. Install Dependencies and Check GPU

`ultralytics`가 PyTorch/YOLO 학습과 검증을 처리합니다. GPU가 잡히면 `device=0`, 아니면 CPU로 실행됩니다.

In [ ]:
!pip -q install "ultralytics>=8.4.0" roboflow pandas matplotlib seaborn opencv-python-headless pyyaml

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import cv2
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"torch={torch.__version__}")
print(f"cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu={torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not available. Training will run on CPU and will be much slower.")
print(f"ultralytics_device={DEVICE}")

## 4. Download Roboflow Dataset

UTS Aircraft Defect Detection v3 데이터셋을 Roboflow Universe에서 YOLOv8 형식으로 다운로드합니다.

In [ ]:
os.chdir(DATASET_DIR)

from roboflow import Roboflow

if not ROBOFLOW_API_KEY or ROBOFLOW_API_KEY.startswith("<PASTE_"):
    raise ValueError("Set ROBOFLOW_API_KEY before downloading the dataset.")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
versions = project.versions()
print("Available versions:", [v.version for v in versions])

if str(ROBOFLOW_VERSION).lower() == "latest":
    version_num = max(int(str(v.version).split("/")[-1]) for v in versions)
else:
    version_num = int(ROBOFLOW_VERSION)

print(f"Selected version: {version_num}")
version = project.version(version_num)
dataset = version.download("yolov8")

DATA_YAML = Path(dataset.location) / "data.yaml"
print(f"Dataset location: {dataset.location}")
print(f"data.yaml: {DATA_YAML}")

## 5. Inspect Dataset

학습 전에 split별 이미지 수와 클래스 순서를 확인합니다. 백엔드는 `Dent`, `Fastener Damage`, `Rupture` 순서의 YOLO detector를 기대합니다.

In [ ]:
def resolve_dataset_path(value):
    p = Path(value)
    if p.is_absolute():
        return p
    return DATA_YAML.parent / p

with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

raw_names = data_cfg.get("names", [])
if isinstance(raw_names, dict):
    class_names = [raw_names[i] for i in sorted(raw_names)]
else:
    class_names = list(raw_names)

expected_classes = ["Dent", "Fastener Damage", "Rupture"]
print("Class names:", class_names)
if class_names != expected_classes:
    print("WARNING: Class order differs from backend expectation:", expected_classes)

image_exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
summary_rows = []
for split in ["train", "val", "valid", "test"]:
    if split not in data_cfg:
        continue
    split_path = resolve_dataset_path(data_cfg[split])
    if split_path.name != "images" and (split_path / "images").exists():
        split_path = split_path / "images"
    count = len([p for p in split_path.rglob("*") if p.suffix.lower() in image_exts]) if split_path.exists() else 0
    summary_rows.append({"split": split, "path": str(split_path), "images": count})

split_df = pd.DataFrame(summary_rows)
display(split_df)

## 6. Train YOLOv8n on Colab GPU

학습 산출물은 `/content/aircraft_defect_training/training/runs/aircraft-v1/` 아래에 저장됩니다.

In [ ]:
train_started_at = time.time()

model = YOLO(BASE_MODEL)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device=DEVICE,
    patience=PATIENCE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    workers=2,
    seed=SEED,
    save_period=-1,
    plots=True,
    verbose=True,
)

train_seconds = time.time() - train_started_at
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

print(f"Training complete in {train_seconds / 60:.1f} minutes")
print(f"RUN_DIR={RUN_DIR}")
print(f"BEST_PT={BEST_PT} exists={BEST_PT.exists()}")
print(f"LAST_PT={LAST_PT} exists={LAST_PT.exists()}")
if BEST_PT.exists():
    print(f"best.pt size={BEST_PT.stat().st_size / 1e6:.1f} MB")

## 7. Validate the Best Model

`best.pt`를 다시 로드해서 검증을 수행합니다. 발표 자료에는 `mAP50`, `mAP50-95`, precision, recall 값을 사용하면 됩니다.

In [ ]:
if not BEST_PT.exists():
    raise FileNotFoundError(f"Missing trained weights: {BEST_PT}")

best_model = YOLO(str(BEST_PT))
val_results = best_model.val(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=f"{RUN_NAME}-val",
    exist_ok=True,
    plots=True,
)

metrics_summary = {
    "precision": float(val_results.box.mp),
    "recall": float(val_results.box.mr),
    "mAP50": float(val_results.box.map50),
    "mAP50-95": float(val_results.box.map),
}
metrics_summary

## 8. Load Training History

Ultralytics가 저장한 `results.csv`를 읽어서 loss와 metric 변화를 차트로 만듭니다.

In [ ]:
RESULTS_CSV = RUN_DIR / "results.csv"
if not RESULTS_CSV.exists():
    raise FileNotFoundError(f"Missing results.csv: {RESULTS_CSV}")

df = pd.read_csv(RESULTS_CSV)
df.columns = [c.strip() for c in df.columns]
if "epoch" not in df.columns:
    df.insert(0, "epoch", range(len(df)))

display(df.tail())
print("columns:")
for c in df.columns:
    print(" -", c)

## 9. Create Presentation Charts

아래 셀은 발표 자료에 바로 넣을 수 있는 PNG 차트를 `/content/presentation_assets/`에 저장합니다.

In [ ]:
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 220
plt.rcParams["font.size"] = 12

ASSET_DIR.mkdir(parents=True, exist_ok=True)

def existing_columns(candidates):
    return [c for c in candidates if c in df.columns]

def savefig(name):
    out_path = ASSET_DIR / name
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"saved: {out_path}")
    return out_path

# 1) Training and validation loss curves
loss_cols = existing_columns([
    "train/box_loss", "train/cls_loss", "train/dfl_loss",
    "val/box_loss", "val/cls_loss", "val/dfl_loss",
])
plt.figure(figsize=(12, 7))
for col in loss_cols:
    plt.plot(df["epoch"], df[col], marker="o", linewidth=2, label=col)
plt.title("YOLOv8n Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(ncol=2, fontsize=10)
savefig("01_training_validation_loss.png")

# 2) Detection metric curves
metric_cols = existing_columns([
    "metrics/precision(B)", "metrics/recall(B)",
    "metrics/mAP50(B)", "metrics/mAP50-95(B)",
])
plt.figure(figsize=(12, 7))
for col in metric_cols:
    plt.plot(df["epoch"], df[col], marker="o", linewidth=2.5, label=col.replace("metrics/", ""))
plt.ylim(0, 1.02)
plt.title("Validation Metrics by Epoch")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.legend(fontsize=10)
savefig("02_validation_metrics.png")

# 3) mAP progress focus chart
map_cols = existing_columns(["metrics/mAP50(B)", "metrics/mAP50-95(B)"])
plt.figure(figsize=(11, 6))
for col in map_cols:
    plt.plot(df["epoch"], df[col], marker="o", linewidth=3, label=col.replace("metrics/", ""))
plt.ylim(0, 1.02)
plt.title("mAP Progress During Training")
plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.legend(fontsize=11)
savefig("03_map_progress.png")

# 4) Final validation metric summary
final_metric_values = {
    "Precision": metrics_summary.get("precision"),
    "Recall": metrics_summary.get("recall"),
    "mAP50": metrics_summary.get("mAP50"),
    "mAP50-95": metrics_summary.get("mAP50-95"),
}
summary_plot_df = pd.DataFrame({
    "metric": list(final_metric_values.keys()),
    "score": [float(v) for v in final_metric_values.values()],
})
plt.figure(figsize=(10, 6))
bar_colors = ["#2563eb", "#059669", "#dc2626", "#7c3aed"]
ax = sns.barplot(data=summary_plot_df, x="metric", y="score", palette=bar_colors, hue="metric", legend=False)
ax.set_ylim(0, 1.02)
ax.set_title("Final Validation Metrics")
ax.set_xlabel("")
ax.set_ylabel("Score")
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=4)
savefig("04_final_metric_summary.png")

## 10. Collect Ultralytics Built-in Plots

Ultralytics가 자동 생성한 confusion matrix, PR curve, F1 curve, batch prediction 이미지를 발표용 폴더로 복사합니다.

In [ ]:
def copy_if_exists(src, dst_name):
    src = Path(src)
    if src.exists():
        dst = ASSET_DIR / dst_name
        shutil.copy2(src, dst)
        print(f"copied: {src.name} -> {dst}")
        return dst
    return None

copied_assets = []
for filename in [
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "labels.jpg",
    "val_batch0_pred.jpg",
    "val_batch1_pred.jpg",
    "val_batch2_pred.jpg",
]:
    copied = copy_if_exists(RUN_DIR / filename, f"ultralytics_{filename}")
    if copied:
        copied_assets.append(copied)

VAL_RUN_DIR = RUNS_DIR / f"{RUN_NAME}-val"
for filename in [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "val_batch0_pred.jpg",
    "val_batch1_pred.jpg",
    "val_batch2_pred.jpg",
]:
    copied = copy_if_exists(VAL_RUN_DIR / filename, f"validation_{filename}")
    if copied:
        copied_assets.append(copied)

print(f"Total copied built-in assets: {len(copied_assets)}")

## 11. Generate Sample Prediction Images

테스트 이미지 몇 장을 골라 bounding box가 그려진 예측 이미지와 contact sheet를 생성합니다.

In [ ]:
def split_images_dir(split_name):
    if split_name not in data_cfg:
        return None
    split_path = resolve_dataset_path(data_cfg[split_name])
    if split_path.name != "images" and (split_path / "images").exists():
        split_path = split_path / "images"
    return split_path if split_path.exists() else None

sample_source_dir = split_images_dir("test") or split_images_dir("val") or split_images_dir("valid")
if sample_source_dir is None:
    raise FileNotFoundError("Could not find test/val image directory")

all_images = sorted([p for p in sample_source_dir.rglob("*") if p.suffix.lower() in image_exts])
random.Random(SEED).shuffle(all_images)
sample_images = all_images[:8]
print(f"Using {len(sample_images)} sample images from {sample_source_dir}")

prediction_dir = ASSET_DIR / "sample_predictions"
prediction_dir.mkdir(parents=True, exist_ok=True)
prediction_paths = []

for idx, img_path in enumerate(sample_images, start=1):
    result = best_model.predict(
        source=str(img_path),
        imgsz=IMGSZ,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        device=DEVICE,
        verbose=False,
    )[0]
    annotated = result.plot()  # BGR numpy array from Ultralytics/OpenCV
    out_path = prediction_dir / f"sample_{idx:02d}_{img_path.stem}.jpg"
    cv2.imwrite(str(out_path), annotated)
    prediction_paths.append(out_path)
    print(f"saved: {out_path}")

# Build a contact sheet for presentation slides.
thumb_w, thumb_h = 520, 360
cols = 2
rows = (len(prediction_paths) + cols - 1) // cols
sheet = Image.new("RGB", (cols * thumb_w, rows * thumb_h), "white")

def fit_image(img, size):
    img = img.copy()
    img.thumbnail(size, Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", size, "white")
    x = (size[0] - img.width) // 2
    y = (size[1] - img.height) // 2
    canvas.paste(img, (x, y))
    return canvas

for i, path in enumerate(prediction_paths):
    img = Image.open(path).convert("RGB")
    tile = fit_image(img, (thumb_w, thumb_h))
    x = (i % cols) * thumb_w
    y = (i // cols) * thumb_h
    sheet.paste(tile, (x, y))

contact_sheet_path = ASSET_DIR / "05_sample_predictions_contact_sheet.png"
sheet.save(contact_sheet_path, quality=95)
print(f"saved: {contact_sheet_path}")
display(sheet)

## 12. Write Training Summary for Slides

발표 자료에 들어갈 핵심 수치를 `training_summary.csv`와 `model_card.md`로 저장합니다.

In [ ]:
training_minutes = train_seconds / 60 if "train_seconds" in globals() else None
summary = {
    "dataset": "UTS Aircraft Defect Detection v3 / Roboflow Universe",
    "model": BASE_MODEL,
    "epochs": EPOCHS,
    "batch": BATCH,
    "image_size": IMGSZ,
    "patience": PATIENCE,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "training_minutes": round(training_minutes, 2) if training_minutes is not None else None,
    "classes": ", ".join(class_names),
    "precision": round(metrics_summary["precision"], 4),
    "recall": round(metrics_summary["recall"], 4),
    "mAP50": round(metrics_summary["mAP50"], 4),
    "mAP50-95": round(metrics_summary["mAP50-95"], 4),
    "best_pt": str(BEST_PT),
    "presentation_assets": str(ASSET_DIR),
}

summary_df = pd.DataFrame([summary])
summary_csv = ASSET_DIR / "training_summary.csv"
summary_df.to_csv(summary_csv, index=False)
display(summary_df)

model_card = f"""# Aircraft Defect Detection Training Summary

- Dataset: {summary['dataset']}
- Model: {summary['model']}
- Classes: {summary['classes']}
- Epochs: {summary['epochs']}
- Batch size: {summary['batch']}
- Image size: {summary['image_size']}
- Device: {summary['device']}
- Training time: {summary['training_minutes']} minutes

## Validation Metrics

| Metric | Score |
|---|---:|
| Precision | {summary['precision']} |
| Recall | {summary['recall']} |
| mAP50 | {summary['mAP50']} |
| mAP50-95 | {summary['mAP50-95']} |

## Output

- Best weights: `{BEST_PT}`
- Presentation assets: `{ASSET_DIR}`
"""
model_card_path = ASSET_DIR / "model_card.md"
model_card_path.write_text(model_card, encoding="utf-8")
print(model_card)
print(f"saved: {summary_csv}")
print(f"saved: {model_card_path}")

## 13. Package and Download Outputs

`best.pt`는 로컬 프로젝트의 `backend/model/best.pt`로 교체하면 됩니다. 발표용 이미지는 ZIP으로 내려받아 슬라이드에 삽입하세요.

In [ ]:
zip_path = Path("/content/aircraft_defect_presentation_assets.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(ASSET_DIR.rglob("*")):
        if path.is_file():
            zf.write(path, path.relative_to(ASSET_DIR.parent))

print(f"Presentation assets ZIP: {zip_path} size={zip_path.stat().st_size / 1e6:.1f} MB")
print(f"Best weights: {BEST_PT} size={BEST_PT.stat().st_size / 1e6:.1f} MB")

# In Colab, uncomment these lines to download directly.
# from google.colab import files
# files.download(str(zip_path))
# files.download(str(BEST_PT))

## 14. Optional: Save to Google Drive

Colab 세션이 종료되면 `/content` 파일은 사라집니다. 오래 보관하려면 아래 셀을 실행해서 Drive로 복사하세요.

In [ ]:
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_out = Path("/content/drive/MyDrive/aircraft_defect_training_outputs")
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BEST_PT, drive_out / "best.pt")
    shutil.copy2(zip_path, drive_out / zip_path.name)
    if RESULTS_CSV.exists():
        shutil.copy2(RESULTS_CSV, drive_out / "results.csv")
    print(f"Saved outputs to {drive_out}")
else:
    print("Set SAVE_TO_DRIVE = True and rerun this cell to copy outputs to Google Drive.")

## 15. Use the Colab-Trained Model Locally

GPU에서 학습한 `best.pt`는 CPU 환경에서도 실행됩니다. 로컬 프로젝트에서는 내려받은 `best.pt`를 아래 위치에 복사하면 됩니다.

```bash
cp best.pt /Users/rapidbear/Desktop/PR/backend/model/best.pt
cd /Users/rapidbear/Desktop/PR/backend
python main.py
```

백엔드는 `ultralytics.YOLO("backend/model/best.pt")`로 모델을 로드합니다. CUDA 없이 CPU만 있어도 로딩과 추론은 가능하지만, 응답 시간이 느려질 수 있습니다.